# AI-Powered Indonesian Music Engine — Skala Penuh (900K+ Lagu)
**Fokus Bidang:** Data Science & Machine Learning — NLP Semantik (Sentence Embeddings) + Approximate Nearest Neighbor (ANN) Search

**Latar Belakang Penelitian:**
Sistem rekomendasi musik konvensional umumnya mengandalkan metadata atau penyaringan kolaboratif yang kerap mengabaikan kedalaman makna naratif lirik lagu. Proyek penelitian ini mengimplementasikan pendekatan berbasis *Natural Language Processing* (NLP) dan pembelajaran mesin untuk mengekstraksi representasi semantik lirik dari korpus musik berbahasa Indonesia berskala besar. Sistem ini mampu mengidentifikasi lagu-lagu yang memiliki kesamaan makna, nuansa emosional, dan konteks cerita dengan akurasi semantik yang tinggi.

**Spesifikasi Teknis:**
1. **Eksplorasi Korpus Skala Penuh:** Memproses korpus global (~955 ribu trek) untuk mengekstrak dan memfilter seluruh representasi lagu berbahasa Indonesia.
2. **Klasifikasi Bahasa Berbasis AI:** Menggunakan arsitektur `fastText` terkalibrasi (`lid.176.bin`) untuk mendeteksi bahasa teks lirik secara deterministik dan presisi tinggi.
3. **Representasi Semantik & Pengindeksan Vektor:** Menggunakan model transformator multibahasa (*Sentence Transformers*) untuk mengonversi lirik ke dalam *dense vector embeddings*, dioptimalkan dengan pustaka FAISS (*Facebook AI Similarity Search*) untuk pencarian *nearest neighbor* berkecepatan sub-milidetik.

In [1]:
# Instalasi seluruh pustaka esensial (Pemrosesan Data, Machine Learning, NLP, dan FAISS)
!pip install -q kagglehub git+https://github.com/facebookresearch/fastText.git sentence-transformers faiss-cpu pyarrow numpy==1.26.4

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import pandas as pd
import os
import kagglehub

# 1. Unduh & muat SELURUH dataset Spotify (tanpa sampling)
path = kagglehub.dataset_download("bwandowando/spotify-songs-with-attributes-and-lyrics")
print(f"Dataset tersimpan di: {path}")

print("Sedang memuat dataset spotify data asli...")
df_mentah = pd.read_csv(os.path.join(path, "songs_with_attributes_and_lyrics.csv"))

# 2. Menginspeksi wujud asli data sebelum direduksi
print("\n--- ARSITEKTUR DATA MENTAH ---")
print(f"Dimensi (Baris, Kolom): {df_mentah.shape}")
print("\nDaftar Kolom Asli:")
for i, kolom in enumerate(df_mentah.columns, 1):
    print(f"{i}. {kolom}")

Using Colab cache for faster access to the 'spotify-songs-with-attributes-and-lyrics' dataset.
Dataset tersimpan di: /kaggle/input/spotify-songs-with-attributes-and-lyrics
Sedang memuat dataset spotify data asli...

--- ARSITEKTUR DATA MENTAH ---
Dimensi (Baris, Kolom): (955320, 17)

Daftar Kolom Asli:
1. id
2. name
3. album_name
4. artists
5. danceability
6. energy
7. key
8. loudness
9. mode
10. speechiness
11. acousticness
12. instrumentalness
13. liveness
14. valence
15. tempo
16. duration_ms
17. lyrics


### Kamus Data (Data Dictionary) Spotify API
Dataset ini terdiri dari 17 variabel yang terbagi menjadi tiga kategori utama: Metadata Identitas, Fitur Analisis Audio (*Audio Features*), dan Data Tekstual.

| Kategori | Nama Kolom | Deskripsi |
| :--- | :--- | :--- |
| **Metadata** | `id` | ID alfanumerik unik Spotify untuk trek lagu. |
| | `name` | Judul lagu. |
| | `album_name` | Nama album dari lagu tersebut. |
| | `artists` | Nama musisi atau grup band. |
| | `duration_ms` | Durasi trek dalam satuan milidetik. |
| **Data Teks** | `lyrics` | Teks lirik lagu utuh hasil ekstraksi. |
| **Fitur Audio** | `valence` | Skor (0.0 - 1.0) tingkat polaritas emosi positif (valensi). Nilai tinggi mencerminkan perasaan bahagia/ceria, sedangkan nilai rendah mencerminkan kesedihan/melankolis. |
| | `acousticness` | Skor (0.0 - 1.0) tingkat keyakinan bahwa trek tersebut murni instrumen akustik tanpa synthesizer elektrik. |
| | `energy` | Skor (0.0 - 1.0) intensitas dan aktivitas perseptual audio (dinamika, ritme, dan kekerasan suara). |
| | `danceability` | Skor (0.0 - 1.0) kesesuaian tempo, keteraturan ketukan, dan stabilitas ritme untuk aktivitas tari/menari. |
| | `instrumentalness` | Prediksi (0.0 - 1.0) ketiadaan vokal. Skor mendekati 1.0 mengindikasikan trek instrumental murni. |
| | `speechiness` | Deteksi keberadaan vokal lisan (seperti *spoken word*, puisi, atau rap). |
| | `liveness` | Deteksi rekaman pertunjukan langsung (*live concert/performance*) bersama penonton versus rekaman studio. |
| | `tempo` | Estimasi kecepatan ritmis trek dalam satuan *Beats Per Minute* (BPM). |
| | `loudness` | Rata-rata tingkat volume suara trek dalam satuan desibel (dB). |
| | `key` | Kunci nada dasar trek menggunakan pemetaan notasi standar *Pitch Class*. |
| | `mode` | Modalitas skala nada trek (Mayor = 1, Minor = 0). |

*Catatan: Pemahaman menyeluruh terhadap kamus data ini menjadi landasan logis dalam tahapan seleksi fitur dan analisis data eksploratif.*

### Karakteristik dan Signifikansi Fitur Audio
Meskipun pencarian semantik berfokus pada konten tekstual lirik, parameter psikoakustik audio seperti *valence*, *acousticness*, *energy*, dan *danceability* tetap dipertahankan. Atribut audio ini berfungsi sebagai profil pendukung untuk memvalidasi karakteristik emosional dan musikalitas trek:

*   **`valence` (Valensi / Polarisasi Emosi):** Mengukur derajat emosi positif yang ditransmisikan oleh musik (rentang 0.0 hingga 1.0). Skor rendah merefleksikan suasana melankolis atau sedih, sementara skor tinggi menunjukkan suasana bahagia atau ceria.
*   **`acousticness` (Indeks Akustik):** Probabilitas keberadaan instrumen akustik alami dibandingkan instrumen elektronik atau sintetis.
*   **`energy` (Intensitas Energi):** Representasi perseptual dari intensitas, dinamika, dan aktivitas frekuensi audio.
*   **`danceability` (Keteraturan Ritmik):** Evaluasi keteraturan tempo, kekuatan ketukan, dan stabilitas ritme untuk kebutuhan ritmis.

Integrasi data audio ini memperkaya dimensi analisis dan memungkinkan visualisasi multi-aspek pada sistem rekomendasi.

In [3]:
# 1. Reduksi 8 kolom target & buang baris tanpa lirik (SELURUH data, tanpa sampling)
kolom_target = ['id', 'name', 'artists', 'lyrics', 'valence', 'acousticness', 'energy', 'danceability']
df_valid = df_mentah[kolom_target].dropna(subset=['lyrics']).copy()
df_valid = df_valid[df_valid['lyrics'].astype(str).str.strip().str.len() > 0]
df_valid.reset_index(drop=True, inplace=True)

print(f"Total lagu mentah          : {df_mentah.shape[0]:,}")
print(f"Total lagu dengan lirik valid (diproses semua, tanpa sampling): {df_valid.shape[0]:,}")
display(df_valid[['name', 'artists']].head(3))

Total lagu mentah          : 955,320
Total lagu dengan lirik valid (diproses semua, tanpa sampling): 955,307


,name,artists
0,!,HELLYEAH
1,!!,Yxngxr1
2,!!! - Interlude,Glowie


In [4]:
# 1. Mengaudit tipe data dan memastikan struktur bersih pada SELURUH data
print("--- PROFIL STRUKTUR DATA (SELURUH DATASET) ---")
df_valid.info()

# 2. Membedah distribusi statistik 4 metrik esensial
print("\n--- DISTRIBUSI STATISTIK FITUR AUDIO ---")
fitur_audio_stats = df_valid[['valence', 'acousticness', 'energy', 'danceability']]
display(fitur_audio_stats.describe())

--- PROFIL STRUKTUR DATA (SELURUH DATASET) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 955307 entries, 0 to 955306
Data columns (total 8 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   id            955307 non-null  object 
 1   name          955296 non-null  object 
 2   artists       955302 non-null  object 
 3   lyrics        955307 non-null  object 
 4   valence       955307 non-null  float64
 5   acousticness  955307 non-null  float64
 6   energy        955307 non-null  float64
 7   danceability  955307 non-null  float64
dtypes: float64(4), object(4)
memory usage: 58.3+ MB

--- DISTRIBUSI STATISTIK FITUR AUDIO ---


,valence,acousticness,energy,danceability
count,955307.000000,955307.000000,955307.000000,955307.000000
mean,0.488119,0.282963,0.652441,0.550710
std,0.251467,0.311801,0.238823,0.169784
min,0.000000,0.000000,0.000000,0.000000
25%,0.282000,0.011900,0.482000,0.436000
50%,0.477000,0.142000,0.687000,0.558000
75%,0.690000,0.518000,0.857000,0.675000
max,1.000000,0.996000,1.000000,0.993000


## Eksplorasi Data: Pengelompokan Berdasarkan Profil Audio (Unsupervised Learning)
Pada tahap eksplorasi awal, algoritma `MiniBatchKMeans` diterapkan untuk mengelompokkan trek musik berdasarkan kemiripan parameter audio (*valence*, *acousticness*, *energy*, dan *danceability*). Pendekatan *clustering* ini memetakan spektrum musikal ke dalam beberapa klaster sentroid.

Meskipun segmentasi klaster memberikan wawasan representatif mengenai distribusi suasana audio (*audio mood distribution*), arsitektur sistem rekomendasi akhir dirancang untuk memproses **seluruh spektrum lagu tanpa batasan klaster**. Hal ini memastikan cakupan katalog lagu berbahasa Indonesia yang komprehensif tanpa kehilangan keberagaman genre musik.

In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import MiniBatchKMeans

# 1. Standarisasi Skala Fitur (Feature Scaling) pada SELURUH dataset
fitur_audio = df_valid[['valence', 'acousticness', 'energy', 'danceability']]
scaler = StandardScaler()
audio_scaled = scaler.fit_transform(fitur_audio)

print(f"Melatih MiniBatchKMeans pada {audio_scaled.shape[0]:,} lagu...")

# 2. Pelatihan Model Clustering
kmeans = MiniBatchKMeans(
    n_clusters=5,          # Mengelompokkan data ke dalam 5 nuansa utama
    random_state=42,       # Memastikan hasil klaster dapat direproduksi secara konsisten
    n_init=10,             # Melakukan 10 kali inisialisasi centroid awal untuk mencari yang paling optimal
    batch_size=10_000,     # Memproses data per 10.000 baris untuk efisiensi memori
)

# Mempelajari distribusi data sekaligus menghasilkan prediksi label klaster
df_valid['klaster_vibes'] = kmeans.fit_predict(audio_scaled)

# 3. Analisis Nilai Tengah (Centroid Analysis)
print("\nKarakteristik Rata-Rata Metrik untuk ke-5 Klaster:")
hasil_klaster = df_valid.groupby('klaster_vibes')[['valence', 'acousticness', 'energy', 'danceability']].mean()
display(hasil_klaster)

Melatih MiniBatchKMeans pada 955,307 lagu...

Karakteristik Rata-Rata Metrik untuk ke-5 Klaster:


,valence,acousticness,energy,danceability
klaster_vibes,,,,
0,0.291033,0.047301,0.837325,0.362884
1,0.550478,0.640286,0.450735,0.619676
2,0.802329,0.202690,0.734743,0.707891
3,0.474923,0.093393,0.739786,0.596282
4,0.234911,0.738580,0.295141,0.413580


### Evaluasi Karakteristik Sentroid Klaster
Matriks sentroid di atas menyajikan nilai rata-rata tiap fitur audio pada masing-masing klaster, mendemonstrasikan bagaimana algoritma memisahkan spektrum suasana lagu (misalnya klaster dengan energi tinggi versus klaster melankolis/akustik).

Pada tahapan pemrosesan berikutnya, pembatasan berbasis klaster dihilangkan. Seluruh entri lagu dari setiap kelompok audio diikutsertakan dalam tahap klasifikasi bahasa dan pemrosesan representasi lirik untuk memaksimalkan retensi korpus musik Indonesia.

In [6]:
# Ekstraksi otomatis klaster Mellow/Balada.
# Karakteristik utama: tingkat kebahagiaan (valence) dan intensitas audio (energy) yang sangat rendah.
# Kita menjumlahkan rata-rata valence dan energy tiap klaster, lalu mencari nilai minimumnya.
skor_mellow = hasil_klaster['valence'] + hasil_klaster['energy']
klaster_mellow_id = skor_mellow.idxmin()

print(f"Klaster bernuansa 'mellow' berhasil dideteksi otomatis pada: Klaster {klaster_mellow_id}")
display(hasil_klaster.loc[[klaster_mellow_id]])

# Memisahkan lagu-lagu di dalam klaster target ke DataFrame baru
df_mellow = df_valid[df_valid['klaster_vibes'] == klaster_mellow_id].copy()
df_mellow.reset_index(drop=True, inplace=True)
print(f"\nTotal lagu pada klaster mellow: {df_mellow.shape[0]:,} lagu (dari total keseluruhan {df_valid.shape[0]:,} lagu).")

Klaster bernuansa 'mellow' berhasil dideteksi otomatis pada: Klaster 4


,valence,acousticness,energy,danceability
klaster_vibes,,,,
4,0.234911,0.73858,0.295141,0.41358



Total lagu pada klaster mellow: 131,302 lagu (dari total keseluruhan 955,307 lagu).


## Identifikasi Bahasa Korpus dan Prapemrosesan Teks NLP
Tahap krusial dalam pipeline data adalah memisahkan korpus berbahasa Indonesia dari dataset global yang beranggotakan hampir satu juta trek multinasional.

Identifikasi bahasa dilakukan secara otomatis menggunakan model inferensi bahasa terkalibrasi **fastText** (`lid.176.bin`), yang mengevaluasi struktur n-gram karakter teks lirik dengan probabilitas tinggi. Proses ini diterapkan secara menyeluruh pada semua kandidat lagu untuk menjamin kelengkapan data.

Setelah subset lagu berbahasa Indonesia berhasil diisolasi, prosedur prapemrosesan teks (*text cleaning*) dilakukan untuk:
1. Menghilangkan artefak struktural metadata (seperti label `[Chorus]`, `[Verse]`, penanda akor, atau keterangan instrumen).
2. Menormalkan tanda baca, karakter non-alfanumerik, dan spasi berlebih.
3. Menghasilkan representasi teks bersih (*cleaned tokens*) yang optimal untuk ekstraksi semantik pada model transformator.

In [7]:
import re
import os
import fasttext

# 1. Pemuatan model identifikasi bahasa fastText
if not os.path.exists('lid.176.bin'):
    !wget -q https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin
model_lang = fasttext.load_model('lid.176.bin')

def deteksi_bahasa_robust(teks):
    t_bersih = str(teks).lower()
    # Buang header/metadata umum ala Genius sebelum diproses
    t_bersih = re.sub(r'^\d+\s*contributors?.*?lyrics', ' ', t_bersih, flags=re.DOTALL)
    t_bersih = re.sub(r'translations.*?(?=\n|\[)', ' ', t_bersih, flags=re.DOTALL)
    t_bersih = re.sub(r'\[.*?\]|\(.*?\)', ' ', t_bersih)
    t_bersih = re.sub(r'[^a-z\s]', ' ', t_bersih)
    t_bersih = ' '.join(t_bersih.split())

    # Ambil jendela teks dari TENGAH lirik, bukan cuma awal
    n = len(t_bersih)
    if n < 15:
        return 'unknown'
    start = max(0, (n // 2) - 150)
    t_input = t_bersih[start:start + 300]

    try:
        labels, probs = model_lang.predict(t_input, k=1)
        pred_label = str(labels[0][0]).lower()
        conf = probs[0]
        if ('id' in pred_label or 'ind' in pred_label or 'ms' in pred_label) and conf > 0.5:
            return 'id'
    except:
        pass

    # Fallback tetap dicek meski fastText "yakin" salah — bukan cuma saat error
    kata_kunci_indo = {'yang','aku','kamu','dan','dengan','bisa','tidak','kau','akan','untuk','mereka','tak','ku','mu','ini','itu','kita'}
    kata_lirik = set(t_bersih.split()[:80])
    if len(kata_lirik.intersection(kata_kunci_indo)) >= 3:
        return 'id'

    return 'other'

def bersihkan_lirik(teks):
    # Membersihkan lirik dari karakter spesial untuk persiapan Sentence Embeddings
    t = str(teks).lower()
    t = re.sub(r'\[.*?\]|\(.*?\)', ' ', t)
    t = re.sub(r'[^a-zàâçéèêëîïôûùüÿñæœ\s]', ' ', t)
    return ' '.join(t.split())

# Menggunakan seluruh dataset tanpa mengandalkan filter klaster
print("Melewati proses klastering... Memuat SELURUH data lagu untuk dideteksi bahasanya.")
df_semua_lagu = df_valid.copy()

print(f"Mengeksekusi deteksi bahasa pada {df_semua_lagu.shape[0]:,} lagu...")
df_semua_lagu['bahasa'] = df_semua_lagu['lyrics'].apply(deteksi_bahasa_robust)

# Filtrasi akhir khusus lagu berbahasa Indonesia
df_indo = df_semua_lagu[df_semua_lagu['bahasa'] == 'id'].copy()
df_indo['lirik_clean'] = df_indo['lyrics'].apply(bersihkan_lirik)
df_indo['lirik_clean'] = df_indo['lirik_clean'].fillna('')
df_indo = df_indo[df_indo['lirik_clean'].str.len() > 20]
df_indo['lirik_bersih'] = df_indo['lirik_clean']
df_indo.reset_index(drop=True, inplace=True)

print(f"\nProses Selesai: {df_indo.shape[0]:,} lagu Indonesia (SELURUH KLASTER) berhasil dikurasi.")
if df_indo.shape[0] > 0:
    display(df_indo[['name', 'artists']].head(15))

Melewati proses klastering... Memuat SELURUH data lagu untuk dideteksi bahasanya.
Mengeksekusi deteksi bahasa pada 955,307 lagu...

Proses Selesai: 1,508 lagu Indonesia (SELURUH KLASTER) berhasil dikurasi.


,name,artists
0,# 1,Slank
1,#eeeaa,Endank Soekamti;CJR
2,'Ego',Ekamatra
3,10 Tahun di Barisan,Over Distortion;Tonggos Darurat
4,12 Inches Of Love - Just Say House Album Mix,Jesse Saunders
5,19 Million Ac,The Spits
6,1904.,YAPH
7,1st Battalion Bugle Call / Fall In,['The Corps of Drums of the 1st Battalion The ...
8,8 Tahun,Adhitia Sofyan
9,A,Altimet


In [8]:
# Diagnostic run: deteksi bahasa di SELURUH df_valid, tanpa filter klaster dulu
df_valid['bahasa'] = df_valid['lyrics'].apply(deteksi_bahasa_robust)
df_id_semua = df_valid[df_valid['bahasa'] == 'id']

print(f"Total lagu Indonesia di SELURUH dataset: {df_id_semua.shape[0]:,}")
print("\nSebaran klaster untuk lagu-lagu Indonesia ini:")
display(df_id_semua['klaster_vibes'].value_counts())

Total lagu Indonesia di SELURUH dataset: 1,508

Sebaran klaster untuk lagu-lagu Indonesia ini:


,count
klaster_vibes,
3,444
2,276
4,274
1,264
0,250


## Pemodelan Semantik Lirik: Sentence Transformers & Pengindeksan FAISS

### Keterbatasan Pendekatan Leksikal Tradisional
Pendekatan pencocokan kata berbasis frekuensi (seperti TF-IDF atau bag-of-words) memiliki keterbatasan fundamental dalam memahami konteks semantik (*lexical gap*). Dua lagu yang menyampaikan emosi atau tema yang identik sering kali menggunakan leksikon yang sama sekali berbeda (contoh: *"hujan membasahi kenangan"* dan *"gerimis mengiringi rindu"*). Metode berbasis leksikal murni gagal menangkap kedekatan makna di antara kedua kalimat tersebut.

### Ekstraksi Vektor Semantik (*Dense Embeddings*)
Untuk mengatasi kendala tersebut, digunakan arsitektur transformator multibahasa `paraphrase-multilingual-MiniLM-L12-v2`. Model ini memetakan teks lirik ke dalam ruang vektor berdimensi tinggi (*high-dimensional dense embedding space*), di mana kedekatan posisi vektor merepresentasikan kesamaan semantik dan tematik antar-lagu.

### Pengindeksan Vektor Skalabel dengan FAISS
Untuk memungkinkan kueri kemiripan yang efisien tanpa komputasi *brute-force* yang lambat, vektor dinormalisasi dan diindeks menggunakan pustaka **FAISS** (*Facebook AI Similarity Search*). Dengan indeks *Inner Product* berbasis kosinus (`IndexFlatIP`), sistem mampu mengeksekusi pencarian lagu dengan kemiripan semantik tertinggi dalam skala sub-milidetik.

In [9]:
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss

# 1. Model embedding multibahasa yang ringan & mendukung Bahasa Indonesia secara native
model_embed = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# 2. Encode SELURUH lirik hasil filter menjadi vektor semantik (batch agar hemat memori)
print(f"Membuat embedding semantik untuk {df_indo.shape[0]:,} lirik...")
embeddings = model_embed.encode(
    df_indo['lirik_bersih'].tolist(),
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
)

# 3. Normalisasi vektor -> inner product di FAISS jadi setara cosine similarity
faiss.normalize_L2(embeddings)

# 4. Bangun index FAISS untuk pencarian tetangga terdekat yang cepat & skalabel
dimensi = embeddings.shape[1]
index_faiss = faiss.IndexFlatIP(dimensi)
index_faiss.add(embeddings)

print(f"Index FAISS siap: {index_faiss.ntotal:,} lagu ter-index, dimensi vektor = {dimensi}.")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Membuat embedding semantik untuk 1,508 lirik...


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Index FAISS siap: 1,508 lagu ter-index, dimensi vektor = 384.


In [10]:
def rekomendasikan_lagu_indo(index, jumlah=3):
    # Mengambil vektor semantik dari lagu referensi
    query_vec = embeddings[index:index + 1]

    # Melakukan pencarian tetangga terdekat di FAISS
    # Menambahkan +1 karena hasil teratas selalu merupakan lagu itu sendiri
    skor, idx_tetangga = index_faiss.search(query_vec, jumlah + 1)

    judul = df_indo.iloc[index]['name']
    artis = df_indo.iloc[index]['artists']
    print(f"Referensi Pencarian: '{judul}' oleh {artis}\n")
    print("Sistem merekomendasikan lagu-lagu berikut berdasarkan kemiripan makna lirik:")

    # Menyaring hasil agar tidak menampilkan lagu referensi
    hasil = [(idx, s) for idx, s in zip(idx_tetangga[0], skor[0]) if idx != index][:jumlah]
    for i, (idx, s) in enumerate(hasil, 1):
        j = df_indo.iloc[idx]['name']
        a = df_indo.iloc[idx]['artists']
        print(f"{i}. {j} - {a} (Skor Kesamaan Semantik: {s:.2f})")

# -- Pengujian Modul Rekomendasi --
print("-" * 50)
target_musisi = "Yura Yunita"

# Pencarian dinamis indeks lagu berdasarkan nama musisi
pencarian = df_indo[df_indo['artists'].str.contains(target_musisi, case=False, na=False)]

if not pencarian.empty:
    index_ditemukan = pencarian.index[0]
    rekomendasikan_lagu_indo(index=index_ditemukan)
else:
    print(f"Musisi '{target_musisi}' tidak ditemukan di dalam dataset akhir berbahasa Indonesia.")
    display(df_indo[['name', 'artists']].head(20))

--------------------------------------------------
Referensi Pencarian: 'Apakah Kamu' oleh Yura Yunita

Sistem merekomendasikan lagu-lagu berikut berdasarkan kemiripan makna lirik:
1. Takdir Dan Waktu - Mega (Skor Kesamaan Semantik: 0.82)
2. Separuh Aku - Noah (Skor Kesamaan Semantik: 0.82)
3. Gurauan Berkasih - Siti Nordiana (Skor Kesamaan Semantik: 0.81)


## Serialisasi dan Penyimpanan Artefak Model
Proses inferensi *sentence embeddings* pada korpus data membutuhkan sumber daya komputasi yang intensif. Oleh karena itu, seluruh hasil representasi data dan indeks pencarian diserialisasi ke dalam bentuk artefak permanen:

1. `df_lagu_mellow_indo.parquet`: Tabel data terstruktur yang memuat metadata, metrik audio, dan teks lirik bersih.
2. `embeddings_lagu_mellow.npy`: Matriks vektor *dense embeddings* yang siap digunakan untuk komputasi jarak.
3. `index_lagu_mellow.faiss`: Struktur indeks pencarian tetangga terdekat terindeks FAISS untuk kebutuhan inferensi *real-time*.

Artefak yang telah disimpan dapat langsung dimuat ke dalam antarmuka aplikasi produksi (Streamlit) tanpa perlu mengulang tahapan pra-komputasi dari awal.

In [11]:
import pickle

faiss.write_index(index_faiss, 'index_lagu_mellow.faiss')
df_indo.to_parquet('df_lagu_mellow_indo.parquet', index=False)
np.save('embeddings_lagu_mellow.npy', embeddings)

print("Tersimpan: index_lagu_mellow.faiss, df_lagu_mellow_indo.parquet, embeddings_lagu_mellow.npy")
print("Lain kali cukup load ketiga file ini via faiss.read_index() / pd.read_parquet() / np.load(),")
print("tanpa perlu mengulang clustering, deteksi bahasa, atau encoding embedding dari nol.")

Tersimpan: index_lagu_mellow.faiss, df_lagu_mellow_indo.parquet, embeddings_lagu_mellow.npy
Lain kali cukup load ketiga file ini via faiss.read_index() / pd.read_parquet() / np.load(),
tanpa perlu mengulang clustering, deteksi bahasa, atau encoding embedding dari nol.
